In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
# Text processing & feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import gensim
from gensim.models import Word2Vec
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer

from sentence_transformers import SentenceTransformer

import pickle
from datetime import datetime

In [2]:
# ============================================================
# PHASE 2: TEXT FEATURE EXTRACTION
# ============================================================

print("="*70)
print("PHASE 2: TEXT FEATURE EXTRACTION")
print("="*70)

# Load processed data
print("\n[1/5] Loading preprocessed data...")
df = pd.read_csv('train_data.csv')
print(f"✓ Loaded {len(df)} reviews")

PHASE 2: TEXT FEATURE EXTRACTION

[1/5] Loading preprocessed data...
✓ Loaded 1918055 reviews


In [3]:
# ============================================================
# FEATURE EXTRACTION METHOD 1: SENTIMENT ANALYSIS
# ============================================================

print("\n" + "="*70)
print("FEATURE EXTRACTION 1: SENTIMENT ANALYSIS")
print("="*70)

def extract_sentiment(text):
    """Extract sentiment scores using TextBlob and VADER"""
    if pd.isna(text) or len(str(text).strip()) == 0:
        return {'polarity': 0, 'subjectivity': 0}
    
    blob = TextBlob(str(text))
    return {
        'polarity': blob.sentiment.polarity,      # -1 to 1
        'subjectivity': blob.sentiment.subjectivity  # 0 to 1
    }

print("Extracting sentiment scores...")
sentiment_scores = df['review_text_processed'].apply(extract_sentiment)
sentiment_df = pd.DataFrame(sentiment_scores.tolist())

print(f"✓ Sentiment analysis complete")
print(f"  - Polarity range: [{sentiment_df['polarity'].min():.2f}, {sentiment_df['polarity'].max():.2f}]")
print(f"  - Subjectivity range: [{sentiment_df['subjectivity'].min():.2f}, {sentiment_df['subjectivity'].max():.2f}]")



FEATURE EXTRACTION 1: SENTIMENT ANALYSIS
Extracting sentiment scores...
✓ Sentiment analysis complete
  - Polarity range: [-1.00, 1.00]
  - Subjectivity range: [0.00, 1.00]


In [4]:
# ============================================================
# FEATURE EXTRACTION METHOD 2: TF-IDF VECTORS
# ============================================================

print("\n" + "="*70)
print("FEATURE EXTRACTION 2: TF-IDF VECTORIZATION")
print("="*70)

print("Computing TF-IDF vectors...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=500,           # Top 500 features
    min_df=5,                   # Appear in at least 5 documents
    max_df=0.8,                 # Appear in max 80% of documents
    ngram_range=(1, 2),         # Unigrams and bigrams
    stop_words='english'
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['review_text_processed'])
print(f"✓ TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  - Vocabulary size: {len(tfidf_vectorizer.get_feature_names_out())}")

# Save vocabulary for later use
tfidf_features = tfidf_vectorizer.get_feature_names_out()
print(f"  - Sample features: {tfidf_features[:10].tolist()}")



FEATURE EXTRACTION 2: TF-IDF VECTORIZATION
Computing TF-IDF vectors...
✓ TF-IDF matrix shape: (1918055, 500)
  - Vocabulary size: 500
  - Sample features: ['ability', 'able', 'absolutely', 'account', 'action', 'actually', 'add', 'adult', 'adventure', 'age']


In [5]:
# ============================================================
# FEATURE EXTRACTION METHOD 3: TOPIC MODELING (LDA)
# ============================================================

print("\n" + "="*70)
print("FEATURE EXTRACTION 3: TOPIC MODELING (LDA)")
print("="*70)

print("Fitting LDA model (this may take a moment)...")
n_topics = 15

lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=10,
    learning_method='online',
    n_jobs=-1
)

lda_matrix = lda_model.fit_transform(tfidf_matrix)
print(f"✓ LDA model fitted")
print(f"  - Number of topics: {n_topics}")
print(f"  - Document-topic matrix shape: {lda_matrix.shape}")

# Display top words per topic
print(f"\n  Top words per topic:")
for topic_idx, topic in enumerate(lda_model.components_[:5]):  # Show first 5 topics
    top_indices = topic.argsort()[-5:][::-1]
    top_words = [tfidf_features[i] for i in top_indices]
    print(f"    Topic {topic_idx}: {', '.join(top_words)}")



FEATURE EXTRACTION 3: TOPIC MODELING (LDA)
Fitting LDA model (this may take a moment)...
✓ LDA model fitted
  - Number of topics: 15
  - Document-topic matrix shape: (1918055, 15)

  Top words per topic:
    Topic 0: book, great, recommend, highly, condition
    Topic 1: helpful, easy, information, guide, jane
    Topic 2: movie, read, book, ago, year ago
    Topic 3: good, recipe, good book, book good, nice
    Topic 4: god, bible, christian, faith, study


In [6]:
# ============================================================
# FEATURE EXTRACTION METHOD 4: WORD2VEC EMBEDDINGS
# ============================================================

print("\n" + "="*70)
print("FEATURE EXTRACTION 4: WORD2VEC EMBEDDINGS")
print("="*70)

print("Training Word2Vec model...")
# Convert text to sentences for Word2Vec
sentences = [text.split() for text in df['review_text_processed']]

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,            # Embedding dimension
    window=5,                   # Context window
    min_count=2,                # Minimum word frequency
    workers=4,
    sg=1                        # Skip-gram model
)

print(f"✓ Word2Vec model trained")
print(f"  - Vocabulary size: {len(w2v_model.wv)}")
print(f"  - Vector dimension: {w2v_model.vector_size}")

# Get document-level embeddings (average word vectors)
def get_doc_embedding(text, model, vector_size=100):
    """Compute document embedding as average of word embeddings"""
    tokens = str(text).split()
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

print("Computing document embeddings (this may take a moment)...")
doc_embeddings = np.array([
    get_doc_embedding(text, w2v_model) 
    for text in df['review_text_processed']
])
print(f"✓ Document embeddings computed")
print(f"  - Shape: {doc_embeddings.shape}")



FEATURE EXTRACTION 4: WORD2VEC EMBEDDINGS
Training Word2Vec model...
✓ Word2Vec model trained
  - Vocabulary size: 336065
  - Vector dimension: 100
Computing document embeddings (this may take a moment)...
✓ Document embeddings computed
  - Shape: (1918055, 100)


In [7]:
# ============================================================
# FEATURE EXTRACTION METHOD 5: ASPECT-BASED FEATURES (OPTIMIZED)
# ============================================================

print("\n" + "="*70)
print("FEATURE EXTRACTION 5: ASPECT-BASED FEATURES (OPTIMIZED)")
print("="*70)

# Define book aspects and related keywords
book_aspects = {
    'plot': ['plot', 'story', 'storyline', 'narrative', 'pace', 'ending', 'twist'],
    'characters': ['character', 'protagonist', 'development', 'personality', 'hero'],
    'writing_style': ['writing', 'style', 'prose', 'author', 'dialogue', 'language', 'description'],
    'setting': ['setting', 'world', 'background', 'scene', 'environment', 'location'],
    'emotion': ['emotional', 'moving', 'touching', 'laugh', 'cry', 'heartfelt', 'powerful'],
    'pacing': ['paced', 'slow', 'fast', 'rushed', 'dragging', 'boring']
}

def extract_aspect_sentiments_batch(texts, ratings):
    """Vectorized extraction - much faster for large batches"""
    aspects_list = []
    
    for text, rating in zip(texts, ratings):
        text_lower = str(text).lower()
        aspects_found = {}
        
        for aspect, keywords in book_aspects.items():
            # Check if any keyword for this aspect appears in text
            found = any(keyword in text_lower for keyword in keywords)
            
            if found:
                # Use rating as primary signal (faster than TextBlob for each review)
                # Rating: 1-5 scale, normalize to -1 to 1
                aspect_sentiment = (rating - 3) / 2  # -1 to 1
                aspects_found[aspect] = np.clip(aspect_sentiment, -1, 1)
            else:
                aspects_found[aspect] = None
        
        aspects_list.append(aspects_found)
    
    return aspects_list

print("Extracting aspect-based sentiments (vectorized)...")
# Process in batches for memory efficiency
batch_size = 50000
aspect_sentiments = []

for i in range(0, len(df), batch_size):
    batch_texts = df['review_text_processed'].iloc[i:i+batch_size].values
    batch_ratings = df['rating'].iloc[i:i+batch_size].values
    batch_aspects = extract_aspect_sentiments_batch(batch_texts, batch_ratings)
    aspect_sentiments.extend(batch_aspects)
    print(f"  ✓ Processed {min(i+batch_size, len(df))}/{len(df)} reviews")

print(f"✓ Aspect sentiments extracted for {len(book_aspects)} aspects")



FEATURE EXTRACTION 5: ASPECT-BASED FEATURES (OPTIMIZED)
Extracting aspect-based sentiments (vectorized)...
  ✓ Processed 50000/1918055 reviews
  ✓ Processed 100000/1918055 reviews
  ✓ Processed 150000/1918055 reviews
  ✓ Processed 200000/1918055 reviews
  ✓ Processed 250000/1918055 reviews
  ✓ Processed 300000/1918055 reviews
  ✓ Processed 350000/1918055 reviews
  ✓ Processed 400000/1918055 reviews
  ✓ Processed 450000/1918055 reviews
  ✓ Processed 500000/1918055 reviews
  ✓ Processed 550000/1918055 reviews
  ✓ Processed 600000/1918055 reviews
  ✓ Processed 650000/1918055 reviews
  ✓ Processed 700000/1918055 reviews
  ✓ Processed 750000/1918055 reviews
  ✓ Processed 800000/1918055 reviews
  ✓ Processed 850000/1918055 reviews
  ✓ Processed 900000/1918055 reviews
  ✓ Processed 950000/1918055 reviews
  ✓ Processed 1000000/1918055 reviews
  ✓ Processed 1050000/1918055 reviews
  ✓ Processed 1100000/1918055 reviews
  ✓ Processed 1150000/1918055 reviews
  ✓ Processed 1200000/1918055 reviews


In [9]:
import os
from tqdm import tqdm
import torch  # Add this to check GPU

# Quick GPU check (run this first in a separate cell)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")  # Should show MX550
else:
    print("CUDA not available—stick to CPU or install CUDA toolkit.")

bert_model = SentenceTransformer('all-MiniLM-L6-v2')
texts = df['review_text'].tolist()
chunk_size = 40000  # Or drop to 20000 if VRAM is tight

total_chunks = (len(texts) + chunk_size - 1) // chunk_size

with tqdm(total=total_chunks, desc="Processing chunks") as pbar:
    for i in range(0, len(texts), chunk_size):
        part_num = i // chunk_size
        filename = f'embeddings_part_{part_num}.npy'
        
        if os.path.exists(filename):
            print(f"Skipping chunk {part_num} ({filename} exists).")
            pbar.update(1)
            continue
        
        print(f"Processing chunk {part_num} ({len(texts[i:i+chunk_size])} texts)...")
        chunk = texts[i:i + chunk_size]
        
        # GPU magic: Use 'cuda' if available, fallback to CPU
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        emb = bert_model.encode(
            chunk, 
            batch_size=32,  # Bump to 64 if GPU loves it
            device=device,  # This is the key line!
            show_progress_bar=True  # For inner chunk progress
        )
        np.save(filename, emb)
        print(f"Saved {filename} ({emb.shape[0]} embeddings).")
        pbar.update(1)

print("All chunks processed or skipped!")

CUDA available: False
CUDA not available—stick to CPU or install CUDA toolkit.


Processing chunks: 100%|█████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 4772.13it/s]

Skipping chunk 0 (embeddings_part_0.npy exists).
Skipping chunk 1 (embeddings_part_1.npy exists).
Skipping chunk 2 (embeddings_part_2.npy exists).
Skipping chunk 3 (embeddings_part_3.npy exists).
Skipping chunk 4 (embeddings_part_4.npy exists).
Skipping chunk 5 (embeddings_part_5.npy exists).
Skipping chunk 6 (embeddings_part_6.npy exists).
Skipping chunk 7 (embeddings_part_7.npy exists).
Skipping chunk 8 (embeddings_part_8.npy exists).
Skipping chunk 9 (embeddings_part_9.npy exists).
Skipping chunk 10 (embeddings_part_10.npy exists).
Skipping chunk 11 (embeddings_part_11.npy exists).
Skipping chunk 12 (embeddings_part_12.npy exists).
Skipping chunk 13 (embeddings_part_13.npy exists).
Skipping chunk 14 (embeddings_part_14.npy exists).
Skipping chunk 15 (embeddings_part_15.npy exists).
Skipping chunk 16 (embeddings_part_16.npy exists).
Skipping chunk 17 (embeddings_part_17.npy exists).
Skipping chunk 18 (embeddings_part_18.npy exists).
Skipping chunk 19 (embeddings_part_19.npy exists).


In [10]:
import glob

# Numeric sort to ensure correct order (0,1,2,...,10,11,...)
parts = sorted(
    glob.glob('embeddings_part_*.npy'),
    key=lambda x: int(x.split('_')[-1].split('.')[0])
)

if parts:  # Only if files exist
    full_emb = np.vstack([np.load(p) for p in parts])
    np.save('bert_embeddings.npy', full_emb)
    print(f"Full embeddings saved: {full_emb.shape}")
else:
    print("No embedding parts found.")


Full embeddings saved: (1918055, 384)


In [11]:

# ============================================================
# AGGREGATE FEATURES BY ITEM (OPTIMIZED WITH GROUPBY)
# ============================================================

print("\n" + "="*70)
print("AGGREGATING FEATURES BY ITEM (OPTIMIZED)")
print("="*70)

# Convert everything to DataFrame for efficient groupby
df_work = df.copy()
df_work['sentiment_polarity'] = sentiment_df['polarity'].values
df_work['sentiment_subjectivity'] = sentiment_df['subjectivity'].values

# Add Word2Vec embedding columns
for i in range(100):
    df_work[f'embedding_{i}'] = doc_embeddings[:, i]

# Add BERT embedding columns (NEW!)
print("Loading BERT embeddings...")
bert_embeddings = np.load('bert_embeddings.npy')
print(f"✓ BERT embeddings loaded: {bert_embeddings.shape}")

for i in range(bert_embeddings.shape[1]):  # Should be 384
    df_work[f'bert_embedding_{i}'] = bert_embeddings[:, i]

# Add topic columns
for i in range(n_topics):
    df_work[f'topic_{i}'] = lda_matrix[:, i]

# Add aspect columns
for aspect in book_aspects.keys():
    aspect_vals = []
    for idx, asp_dict in enumerate(aspect_sentiments):
        aspect_vals.append(asp_dict.get(aspect))
    df_work[f'aspect_{aspect}'] = aspect_vals

print("Using groupby for efficient aggregation...")

# Aggregate using groupby
agg_dict = {
    'rating': 'mean',
    'sentiment_polarity': 'mean',
    'sentiment_subjectivity': 'mean',
}

# Add Word2Vec embeddings to aggregation
for i in range(100):
    agg_dict[f'embedding_{i}'] = 'mean'

# Add BERT embeddings to aggregation (NEW!)
for i in range(bert_embeddings.shape[1]):
    agg_dict[f'bert_embedding_{i}'] = 'mean'

# Add topics to aggregation
for i in range(n_topics):
    agg_dict[f'topic_{i}'] = 'mean'

# Add aspects to aggregation
for aspect in book_aspects.keys():
    agg_dict[f'aspect_{aspect}'] = lambda x: np.nanmean(x)

item_feature_matrix = df_work.groupby('item_id').agg(agg_dict).reset_index()
item_feature_matrix = item_feature_matrix.rename(columns={'rating': 'avg_rating'})
item_feature_matrix['review_count'] = df_work.groupby('item_id').size().values

print(f"✓ Item feature matrix created")
print(f"  - Shape: {item_feature_matrix.shape}")
print(f"  - Features per item: {item_feature_matrix.shape[1] - 1}")
print(f"  - Word2Vec features: 100")
print(f"  - BERT features: {bert_embeddings.shape[1]}")


AGGREGATING FEATURES BY ITEM (OPTIMIZED)
Loading BERT embeddings...
✓ BERT embeddings loaded: (1918055, 384)
Using groupby for efficient aggregation...
✓ Item feature matrix created
  - Shape: (199368, 510)
  - Features per item: 509
  - Word2Vec features: 100
  - BERT features: 384


In [12]:
# ============================================================
# CREATE USER PROFILES (OPTIMIZED WITH GROUPBY)
# ============================================================

print("\n" + "="*70)
print("CREATING USER PROFILES (OPTIMIZED)")
print("="*70)

user_agg_dict = {
    'rating': ['mean', 'std'],
    'sentiment_polarity': 'mean',
    'sentiment_subjectivity': 'mean',
}

# Add user embeddings and topics
for i in range(100):
    user_agg_dict[f'embedding_{i}'] = 'mean'
for i in range(n_topics):
    user_agg_dict[f'topic_{i}'] = 'mean'

user_profile_matrix = df_work.groupby('user_id').agg(user_agg_dict).reset_index()

# Flatten column names
user_profile_matrix.columns = ['user_id', 'user_mean_rating', 'user_std_rating', 
                               'user_sentiment_polarity', 'user_sentiment_subjectivity'] + \
                              [f'user_embedding_{i}' for i in range(100)] + \
                              [f'user_topic_{i}' for i in range(n_topics)]

# Fill NaN std values
user_profile_matrix['user_std_rating'] = user_profile_matrix['user_std_rating'].fillna(0)

# Add review count
user_profile_matrix['user_review_count'] = df_work.groupby('user_id').size().values

print(f"✓ User profile matrix created")
print(f"  - Shape: {user_profile_matrix.shape}")
print(f"  - Features per user: {user_profile_matrix.shape[1] - 1}")


CREATING USER PROFILES (OPTIMIZED)
✓ User profile matrix created
  - Shape: (863033, 121)
  - Features per user: 120


In [13]:

# ============================================================
# SAVE ALL FEATURES (CSV METHOD)
# ============================================================
import os

print("\n" + "="*70)
print("SAVING FEATURE MATRICES")
print("="*70)

print("Saving feature matrices (this may take a few minutes)...")

item_feature_matrix.to_csv('item_features.csv', index=False)
print(f"✓ item_features.csv")

user_profile_matrix.to_csv('user_profiles.csv', index=False)
print(f"✓ user_profiles.csv")

df_work.to_csv('train_data_with_indices.csv', index=False)
print(f"✓ train_data_with_indices.csv")

# Save models
print("\nSaving trained models...")

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print(f"✓ tfidf_vectorizer.pkl")

with open('lda_model.pkl', 'wb') as f:
    pickle.dump(lda_model, f)
print(f"✓ lda_model.pkl")

with open('w2v_model.pkl', 'wb') as f:
    pickle.dump(w2v_model, f)
print(f"✓ w2v_model.pkl")

# Save numpy arrays
print("\nSaving feature arrays...")

np.save('tfidf_matrix.npy', tfidf_matrix.toarray())
print(f"✓ tfidf_matrix.npy")

np.save('lda_matrix.npy', lda_matrix)
print(f"✓ lda_matrix.npy")

np.save('doc_embeddings.npy', doc_embeddings)
print(f"✓ doc_embeddings.npy")

# Check BERT embeddings (MOVED HERE!)
if os.path.exists('bert_embeddings.npy'):
    print(f"✓ bert_embeddings.npy (already saved during encoding)")
else:
    print(f"⚠️ WARNING: bert_embeddings.npy not found!")
    print(f"   Make sure BERT encoding cell completed successfully")

print(f"\n✓ All files saved successfully!")


SAVING FEATURE MATRICES
Saving feature matrices (this may take a few minutes)...
✓ item_features.csv
✓ user_profiles.csv
✓ train_data_with_indices.csv

Saving trained models...
✓ tfidf_vectorizer.pkl
✓ lda_model.pkl
✓ w2v_model.pkl

Saving feature arrays...
✓ tfidf_matrix.npy
✓ lda_matrix.npy
✓ doc_embeddings.npy
✓ bert_embeddings.npy (already saved during encoding)

✓ All files saved successfully!


In [14]:

# ============================================================
# FEATURE SUMMARY STATISTICS
# ============================================================

print("\n" + "="*70)
print("FEATURE SUMMARY STATISTICS")
print("="*70)

print(f"\n Item Features:")
print(f"  - Unique items: {len(item_feature_matrix)}")
print(f"  - Avg rating range: [{item_feature_matrix['avg_rating'].min():.2f}, {item_feature_matrix['avg_rating'].max():.2f}]")
print(f"  - Review count range: [{item_feature_matrix['review_count'].min():.0f}, {item_feature_matrix['review_count'].max():.0f}]")
print(f"  - Sentiment polarity range: [{item_feature_matrix['sentiment_polarity'].min():.2f}, {item_feature_matrix['sentiment_polarity'].max():.2f}]")

print(f"\n👤 User Profiles:")
print(f"  - Unique users: {len(user_profile_matrix)}")
print(f"  - Avg user rating range: [{user_profile_matrix['user_mean_rating'].min():.2f}, {user_profile_matrix['user_mean_rating'].max():.2f}]")
print(f"  - Review count per user range: [{user_profile_matrix['user_review_count'].min():.0f}, {user_profile_matrix['user_review_count'].max():.0f}]")

print(f"\n📚 Text Feature Methods:")
print(f"  ✓ Sentiment Analysis (polarity + subjectivity)")
print(f"  ✓ TF-IDF Vectors ({len(tfidf_features)} features)")
print(f"  ✓ Topic Modeling ({n_topics} topics)")
print(f"  ✓ Word2Vec Embeddings (100-dim vectors)")
print(f"  ✓ BERT Embeddings ({bert_embeddings.shape[1]}-dim vectors)")  # NEW!
print(f"  ✓ Aspect-Based Features ({len(book_aspects)} aspects)")

print("\n" + "="*70)
print("PHASE 2 COMPLETE!")
print("="*70)
print("\nGenerated files ready for Phase 3 (Hybrid Recommendation):")
print("  ✓ item_features.csv")
print("  ✓ user_profiles.csv")
print("  ✓ All trained models (.pkl files)")
print("\nNext: Build hybrid recommender system combining these features!")


FEATURE SUMMARY STATISTICS

 Item Features:
  - Unique items: 199368
  - Avg rating range: [1.00, 5.00]
  - Review count range: [1, 2949]
  - Sentiment polarity range: [-1.00, 1.00]

👤 User Profiles:
  - Unique users: 863033
  - Avg user rating range: [1.00, 5.00]
  - Review count per user range: [1, 4420]

📚 Text Feature Methods:
  ✓ Sentiment Analysis (polarity + subjectivity)
  ✓ TF-IDF Vectors (500 features)
  ✓ Topic Modeling (15 topics)
  ✓ Word2Vec Embeddings (100-dim vectors)
  ✓ BERT Embeddings (384-dim vectors)
  ✓ Aspect-Based Features (6 aspects)

PHASE 2 COMPLETE!

Generated files ready for Phase 3 (Hybrid Recommendation):
  ✓ item_features.csv
  ✓ user_profiles.csv
  ✓ All trained models (.pkl files)

Next: Build hybrid recommender system combining these features!
